In [1]:
%pip install statsmodels

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt

# 设置中文字体
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Source Han Sans CN']


Note: you may need to restart the kernel to use updated packages.


In [2]:
# 1.导入数据
print("=" * 50)
print("数据导入与基本信息")
print("=" * 50)

df = pd.read_csv('house_price_train_dataset.csv')

# 数据基本信息
print(f"数据集形状: {df.shape}")
print(f"样本数量: {len(df)}")
print(f"特征数量: {len(df.columns)}")

print("\n数据前5行:")
print(df.head())

print("\n数据集列名:")
print(df.columns.tolist())

print("\n数据基本信息:")
df.info()

数据导入与基本信息
数据集形状: (103871, 88)
样本数量: 103871
特征数量: 88

数据前5行:
          Price    lnPrice  decoration_精装  decoration_简装  decoration_毛坯  \
0  6.194049e+06  15.639100            1.0            0.0            0.0   
1  4.354153e+06  15.286641            1.0            0.0            0.0   
2  3.321992e+06  15.016075            0.0            1.0            0.0   
3  7.895656e+06  15.881823            1.0            0.0            0.0   
4  1.902960e+06  14.458921            1.0            0.0            0.0   

   decoration_其他   总楼层  准确楼层    area  室  ...  city_dist_2  city_dist_3  \
0            0.0   5.0   2.0   52.30  2  ...          0.0          0.0   
1            0.0   6.0   6.0  127.44  3  ...          0.0          0.0   
2            0.0   6.0   1.0  118.02  3  ...          0.0          0.0   
3            0.0   2.0   1.0  293.23  6  ...          0.0          0.0   
4            0.0  10.0   5.0   39.85  0  ...          0.0          0.0   

   city_dist_4  city_dist_5  city_dist_6  ci

In [4]:
# 2.1 OLS模型建立
# 添加必要的库导入
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm

print("=" * 80)
print("房屋价格预测模型 - OLS回归分析")
print("=" * 80)

# 确定自变量和因变量
print("设置变量...")
# 因变量
y_var = 'lnPrice'

# 自变量
X_vars = [col for col in df.columns if col not in ['Price', 'lnPrice','城市','区县','板块']]

print(f"因变量: {y_var}")
print(f"自变量数量: {len(X_vars)}")
print(f"样本数量: {len(df)}")

# 检查变量是否存在
if y_var not in df.columns:
    print(f"错误：因变量 '{y_var}' 不存在于数据集中")
    exit()

# 显示前10个自变量
print(f"\n前10个自变量: {X_vars[:10]}")

# 准备数据
X = df[X_vars]
y = df[y_var]

# 处理缺失值
print(f"\n处理缺失值...")
X_filled = X.copy()
y_filled = y.copy()

# 填充自变量的缺失值
for col in X_filled.columns:
    if X_filled[col].isnull().sum() > 0:
        null_count = X_filled[col].isnull().sum()
        if X_filled[col].dtype in ['float64', 'int64']:
            median_val = X_filled[col].median()
            X_filled[col] = X_filled[col].fillna(median_val)
            print(f"  - {col}: {null_count} 个缺失值已用中位数填充")
        else:
            mode_val = X_filled[col].mode()[0] if not X_filled[col].mode().empty else 0
            X_filled[col] = X_filled[col].fillna(mode_val)
            print(f"  - {col}: {null_count} 个缺失值已用众数填充")

# 填充因变量的缺失值（如果有）
if y_filled.isnull().sum() > 0:
    null_count = y_filled.isnull().sum()
    median_y = y_filled.median()
    y_filled = y_filled.fillna(median_y)
    print(f"  - {y_var}: {null_count} 个缺失值已用中位数填充")

print(f"处理缺失值后数据形状: {X_filled.shape}")

# 划分训练集和测试集
print("\n划分训练集和测试集...")
X_train, X_test, y_train, y_test = train_test_split(
    X_filled, y_filled, test_size=0.2, random_state=42
)
print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 添加常数项
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

# 建立OLS模型
print("\n正在拟合OLS模型...")
ols_model = sm.OLS(y_train, X_train_const)
ols_results = ols_model.fit()

# 样本内预测
y_train_pred = ols_results.predict(X_train_const)

# 样本外预测
y_test_pred = ols_results.predict(X_test_const)

# 计算性能指标
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

# 样本内性能（对数尺度）
train_mae_ln, train_rmse_ln, train_r2_ln = calculate_metrics(y_train, y_train_pred)

# 样本外性能（对数尺度）
test_mae_ln, test_rmse_ln, test_r2_ln = calculate_metrics(y_test, y_test_pred)

# 转换回原始价格尺度进行评估
y_train_true_price = np.exp(y_train)
y_train_pred_price = np.exp(y_train_pred)
y_test_true_price = np.exp(y_test)
y_test_pred_price = np.exp(y_test_pred)

# 原始价格尺度的性能
train_mae_price, train_rmse_price, train_r2_price = calculate_metrics(y_train_true_price, y_train_pred_price)
test_mae_price, test_rmse_price, test_r2_price = calculate_metrics(y_test_true_price, y_test_pred_price)

# 6折交叉验证
print("\n进行6折交叉验证...")
kf = KFold(n_splits=6, shuffle=True, random_state=42)
cv_scores_mae_ln = []
cv_scores_rmse_ln = []
cv_scores_r2_ln = []
cv_scores_mae_price = []
cv_scores_rmse_price = []
cv_scores_r2_price = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_filled), 1):
    # 分割数据
    X_cv_train, X_cv_val = X_filled.iloc[train_idx], X_filled.iloc[val_idx]
    y_cv_train, y_cv_val = y_filled.iloc[train_idx], y_filled.iloc[val_idx]
    
    # 添加常数项
    X_cv_train_const = sm.add_constant(X_cv_train)
    X_cv_val_const = sm.add_constant(X_cv_val)
    
    # 训练模型
    cv_model = sm.OLS(y_cv_train, X_cv_train_const).fit()
    
    # 预测（对数尺度）
    y_cv_pred_ln = cv_model.predict(X_cv_val_const)
    
    # 计算对数尺度指标
    cv_mae_ln, cv_rmse_ln, cv_r2_ln = calculate_metrics(y_cv_val, y_cv_pred_ln)
    cv_scores_mae_ln.append(cv_mae_ln)
    cv_scores_rmse_ln.append(cv_rmse_ln)
    cv_scores_r2_ln.append(cv_r2_ln)
    
    # 计算原始价格尺度指标
    y_cv_true_price = np.exp(y_cv_val)
    y_cv_pred_price = np.exp(y_cv_pred_ln)
    cv_mae_price, cv_rmse_price, cv_r2_price = calculate_metrics(y_cv_true_price, y_cv_pred_price)
    cv_scores_mae_price.append(cv_mae_price)
    cv_scores_rmse_price.append(cv_rmse_price)
    cv_scores_r2_price.append(cv_r2_price)
    
    print(f"  折叠 {fold}: R²(ln) = {cv_r2_ln:.4f}, R²(price) = {cv_r2_price:.4f}")

# 计算交叉验证平均性能
cv_mae_ln_mean = np.mean(cv_scores_mae_ln)
cv_rmse_ln_mean = np.mean(cv_scores_rmse_ln)
cv_r2_ln_mean = np.mean(cv_scores_r2_ln)
cv_mae_price_mean = np.mean(cv_scores_mae_price)
cv_rmse_price_mean = np.mean(cv_scores_rmse_price)
cv_r2_price_mean = np.mean(cv_scores_r2_price)

# 输出模型结果
print("\n" + "=" * 80)
print("OLS模型结果汇总")
print("=" * 80)
print(ols_results.summary())

# 输出性能指标表格
print("\n" + "=" * 80)
print("模型性能指标")
print("=" * 80)

print("\n对数尺度性能指标:")
print("-" * 60)
print(f"{'Metrics':<15} {'In sample':<12} {'Out of sample':<14} {'Cross-validation':<18}")
print("-" * 60)
print(f"{'R²':<15} {train_r2_ln:.4f}{'':<8} {test_r2_ln:.4f}{'':<10} {cv_r2_ln_mean:.4f}")
print(f"{'MAE':<15} {train_mae_ln:.4f}{'':<8} {test_mae_ln:.4f}{'':<10} {cv_mae_ln_mean:.4f}")
print(f"{'RMSE':<15} {train_rmse_ln:.4f}{'':<8} {test_rmse_ln:.4f}{'':<10} {cv_rmse_ln_mean:.4f}")
print("-" * 60)

print("\n原始价格尺度性能指标:")
print("-" * 60)
print(f"{'Metrics':<15} {'In sample':<12} {'Out of sample':<14} {'Cross-validation':<18}")
print("-" * 60)
print(f"{'R²':<15} {train_r2_price:.4f}{'':<8} {test_r2_price:.4f}{'':<10} {cv_r2_price_mean:.4f}")
print(f"{'MAE':<15} {train_mae_price:.2f}{'':<8} {test_mae_price:.2f}{'':<10} {cv_mae_price_mean:.2f}")
print(f"{'RMSE':<15} {train_rmse_price:.2f}{'':<8} {test_rmse_price:.2f}{'':<10} {cv_rmse_price_mean:.2f}")
print("-" * 60)

# 输出详细的统计信息
print(f"\n详细统计信息:")
print(f"样本数量 (训练集): {len(X_train)}")
print(f"样本数量 (测试集): {len(X_test)}")
print(f"自变量数量: {X_train.shape[1]}")
print(f"模型F统计量: {ols_results.fvalue:.2f}")
print(f"F统计量p值: {ols_results.f_pvalue:.4f}")
print(f"AIC: {ols_results.aic:.2f}")
print(f"BIC: {ols_results.bic:.2f}")
print(f"调整R²: {ols_results.rsquared_adj:.4f}")

# 检查多重共线性（VIF）
print(f"\n检查多重共线性...")
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 计算前10个变量的VIF（避免计算所有变量以节省时间）
vif_data = pd.DataFrame()
vif_data["Variable"] = X_train_const.columns[:min(10, len(X_train_const.columns))]
vif_data["VIF"] = [variance_inflation_factor(X_train_const.values, i) 
                   for i in range(min(10, len(X_train_const.columns)))]
print("前10个变量的VIF值:")
print(vif_data)

# 保存模型结果
model_results = {
    'OLS': {
        'model': ols_results,
        'train_r2_ln': train_r2_ln,
        'test_r2_ln': test_r2_ln,
        'cv_r2_ln': cv_r2_ln_mean,
        'train_r2_price': train_r2_price,
        'test_r2_price': test_r2_price,
        'cv_r2_price': cv_r2_price_mean,
        'variables_used': X_vars
    }
}

print("\n" + "=" * 80)
print("OLS回归分析完成")
print("=" * 80)

房屋价格预测模型 - OLS回归分析
设置变量...
因变量: lnPrice
自变量数量: 83
样本数量: 103871

前10个自变量: ['decoration_精装', 'decoration_简装', 'decoration_毛坯', 'decoration_其他', '总楼层', '准确楼层', 'area', '室', '厅', '厨']

处理缺失值...
处理缺失值后数据形状: (103871, 83)

划分训练集和测试集...
训练集大小: (83096, 83)
测试集大小: (20775, 83)

正在拟合OLS模型...

进行6折交叉验证...
  折叠 1: R²(ln) = 0.8190, R²(price) = 0.7019
  折叠 2: R²(ln) = 0.8276, R²(price) = 0.7294
  折叠 3: R²(ln) = 0.8226, R²(price) = 0.7259
  折叠 4: R²(ln) = 0.8166, R²(price) = 0.6931
  折叠 5: R²(ln) = 0.8175, R²(price) = 0.6958
  折叠 6: R²(ln) = 0.8173, R²(price) = 0.6876

OLS模型结果汇总
                            OLS Regression Results                            
Dep. Variable:                lnPrice   R-squared:                       0.821
Model:                            OLS   Adj. R-squared:                  0.820
Method:                 Least Squares   F-statistic:                     4746.
Date:                Wed, 05 Nov 2025   Prob (F-statistic):               0.00
Time:                        23:20:5

In [6]:
# 2.2 OLS预测部分
# 测试集预测部分
def predict_test_set(model_results, test_data_path='house_price_test_dataset.csv'):
    print("\n" + "=" * 80)
    print("测试集预测")
    print("=" * 80)
    
    # 读取测试集数据
    test_df = pd.read_csv(test_data_path)
    print(f"原始测试集形状: {test_df.shape}")
    
    # 获取训练集的特征列表
    ols_model = model_results['OLS']['model']
    train_features = ols_model.model.exog_names
    train_features_order = train_features[1:]  # 排除常数项
    
    print(f"训练模型使用的特征数量: {len(train_features_order)}")
    
    # 1. 特征工程：创建缺失的衍生变量（与训练集保持一致）
    print("\n执行特征工程...")
    
    # 1.1 检查并创建缺失的平方项
    print("检查并创建平方项...")
    square_terms_to_create = ['area^2']  # 根据你的变量列表，只需要创建area^2
    
    for term in square_terms_to_create:
        if term not in test_df.columns and 'area' in test_df.columns:
            test_df[term] = test_df['area'] ** 2
            print(f"已创建平方项: {term}")
    
    # 1.2 检查并创建缺失的城市哑变量
    print("检查并创建城市哑变量...")
    city_dummies_to_create = [f'city_{i}' for i in range(12)]
    
    for city_dummy in city_dummies_to_create:
        if city_dummy not in test_df.columns and '城市' in test_df.columns:
            city_num = int(city_dummy.split('_')[1])
            test_df[city_dummy] = (test_df['城市'] == city_num).astype(int)
            print(f"已创建城市哑变量: {city_dummy}")
    
    # 1.3 检查并创建缺失的城市距离变量
    print("检查并创建城市距离变量...")
    city_dist_to_create = [f'city_dist_{i}' for i in range(12)]
    
    for city_dist in city_dist_to_create:
        if city_dist not in test_df.columns and '城市' in test_df.columns and '距市中心距离_km' in test_df.columns:
            city_num = int(city_dist.split('_')[2])
            test_df[city_dist] = 0
            test_df.loc[test_df['城市'] == city_num, city_dist] = test_df.loc[test_df['城市'] == city_num, '距市中心距离_km']
            print(f"已创建城市距离变量: {city_dist}")
    
    # 1.4 检查并创建缺失的交互项
    print("检查并创建交互项...")
    # 城市和面积的交互项
    city_columns = [f'city_{i}' for i in range(12) if f'city_{i}' in test_df.columns]
    if city_columns and 'area' in test_df.columns:
        for city_col in city_columns:
            interaction_col = f'city_area_interaction_{city_col}'
            if interaction_col not in test_df.columns:
                test_df[interaction_col] = test_df[city_col] * test_df['area']
                print(f"已创建交互项: {interaction_col}")
    
    # 城市和面积平方的交互项
    if city_columns and 'area^2' in test_df.columns:
        for city_col in city_columns:
            interaction_col = f'city_area_sq_interaction_{city_col}'
            if interaction_col not in test_df.columns:
                test_df[interaction_col] = test_df[city_col] * test_df['area^2']
                print(f"已创建交互项: {interaction_col}")
    
    # 2. 处理缺失值（使用训练集的统计量）
    print("\n处理缺失值...")
    
    # 2.1 处理连续变量的缺失值
    continuous_vars = [
        'area', 'building_age', 'greening_rate', 'plot_ratio', 
        'property_fee_avg', 'gas_fee_avg', 'heating_fee_avg', 'parking_spots',
        '总楼层', '准确楼层', '梯数', '户数', '梯户比例_数值', 'building_total',
        'household_total', '距市中心距离_km'
    ]
    
    for var in continuous_vars:
        if var in test_df.columns and test_df[var].isnull().sum() > 0:
            null_count = test_df[var].isnull().sum()
            # 使用训练集的中位数填充
            if var in df.columns:
                train_median = df[var].median()
                test_df[var] = test_df[var].fillna(train_median)
                print(f"  - {var}: {null_count} 个缺失值已用训练集中位数填充")
            else:
                # 如果训练集中没有该变量，使用测试集的中位数
                test_median = test_df[var].median()
                test_df[var] = test_df[var].fillna(test_median)
                print(f"  - {var}: {null_count} 个缺失值已用测试集中位数填充")
    
    # 2.2 处理分类变量和哑变量的缺失值
    categorical_vars = [
        'decoration_精装', 'decoration_简装', 'decoration_毛坯', 'decoration_其他',
        'elevator_yes', 'subway', 'property_phone_yes',
        'south_dummy', 'north_south_dummy',
        'transaction_commercial', 'transaction_non_commercial',
        'usage_commercial_office', 'usage_commercial_residential', 
        'usage_high_end_residential', 'usage_ordinary_residential',
        'house_age_over_2_years', 'house_age_over_5_years', 'house_age_under_2_years',
        'water_civil', 'water_commercial', 'heating_central', 'heating_self',
        'electricity_civil', 'electricity_commercial'
    ]
    
    # 添加城市哑变量
    categorical_vars.extend([f'city_{i}' for i in range(12)])
    
    for var in categorical_vars:
        if var in test_df.columns and test_df[var].isnull().sum() > 0:
            null_count = test_df[var].isnull().sum()
            # 使用训练集的众数填充
            if var in df.columns:
                train_mode = df[var].mode()[0] if not df[var].mode().empty else 0
                test_df[var] = test_df[var].fillna(train_mode)
                print(f"  - {var}: {null_count} 个缺失值已用训练集众数填充")
            else:
                # 如果训练集中没有该变量，使用测试集的众数
                test_mode = test_df[var].mode()[0] if not test_df[var].mode().empty else 0
                test_df[var] = test_df[var].fillna(test_mode)
                print(f"  - {var}: {null_count} 个缺失值已用测试集众数填充")
    
    # 2.3 重新创建衍生变量（确保使用填充后的数据）
    print("\n重新创建衍生变量（使用填充后的数据）...")
    
    # 重新创建平方项
    for term in square_terms_to_create:
        if term in test_df.columns and 'area' in test_df.columns:
            test_df[term] = test_df['area'] ** 2
    
    # 重新创建交互项
    if city_columns and 'area' in test_df.columns:
        for city_col in city_columns:
            interaction_col = f'city_area_interaction_{city_col}'
            if interaction_col in test_df.columns:
                test_df[interaction_col] = test_df[city_col] * test_df['area']
    
    if city_columns and 'area^2' in test_df.columns:
        for city_col in city_columns:
            interaction_col = f'city_area_sq_interaction_{city_col}'
            if interaction_col in test_df.columns:
                test_df[interaction_col] = test_df[city_col] * test_df['area^2']
    
    # 2.4 最终检查是否还有缺失值
    print("\n最终检查是否还有缺失值...")
    remaining_missing = test_df.isnull().sum().sum()
    if remaining_missing > 0:
        print(f"警告: 测试集仍有 {remaining_missing} 个缺失值")
        missing_cols = test_df.isnull().sum()
        missing_cols = missing_cols[missing_cols > 0]
        print("仍有缺失值的列:")
        for col, count in missing_cols.items():
            print(f"- {col}: {count} 个缺失值")
        
        # 对于仍有缺失值的列，使用0填充
        for col in missing_cols.index:
            test_df[col] = test_df[col].fillna(0)
            print(f"已将 {col} 的缺失值填充为0")
    else:
        print("✅ 测试集已无缺失值")
    
    # 3. 确保测试集包含所有训练模型的特征
    print("\n确保特征一致性...")
    
    # 检查缺失的特征并创建它们（设为0）
    missing_features = set(train_features_order) - set(test_df.columns)
    if missing_features:
        print(f"创建缺失的特征并设为0: {len(missing_features)} 个")
        for feature in missing_features:
            test_df[feature] = 0
            print(f"已创建并设为0: {feature}")
    
    # 确保测试集特征顺序与训练模型一致
    test_features_ordered = [feat for feat in train_features_order if feat in test_df.columns]
    
    print(f"测试集实际使用的特征数量: {len(test_features_ordered)}")
    
    # 4. 进行预测
    print("\n使用OLS模型进行预测...")
    
    # 添加常数项
    X_test_final = test_df[test_features_ordered]
    X_test_final_const = sm.add_constant(X_test_final, has_constant='add')
    
    # 检查预测前的数据
    print(f"预测数据形状: {X_test_final_const.shape}")
    print(f"模型参数数量: {len(ols_model.params)}")
    
    try:
        y_test_pred_ln = ols_model.predict(X_test_final_const)
        
        # 检查预测结果是否有空值
        if pd.isna(y_test_pred_ln).any():
            print(f"警告: 预测结果中有 {pd.isna(y_test_pred_ln).sum()} 个空值")
            # 如果有空值，使用中位数填充
            median_pred = np.nanmedian(y_test_pred_ln)
            y_test_pred_ln = np.where(pd.isna(y_test_pred_ln), median_pred, y_test_pred_ln)
            print(f"已将空值填充为预测值中位数: {median_pred:.4f}")
        
        # 创建只包含ID和预测价格的结果DataFrame
        # 假设测试集中有'ID'列，如果没有，使用索引作为ID
        if 'ID' in test_df.columns:
            result_df = pd.DataFrame({
                'ID': test_df['ID'],
                'Price': np.exp(y_test_pred_ln)  # 将lnPrice转换回原始价格
            })
        else:
            result_df = pd.DataFrame({
                'ID': test_df.index,
                'Price': np.exp(y_test_pred_ln)  # 将lnPrice转换回原始价格
            })
        
        # 检查最终结果是否有空值
        if result_df['Price'].isnull().any():
            print(f"警告: 最终结果中有 {result_df['Price'].isnull().sum()} 个空值")
            # 如果有空值，使用中位数填充
            median_price = result_df['Price'].median()
            result_df['Price'] = result_df['Price'].fillna(median_price)
            print(f"已将空值填充为价格中位数: {median_price:.2f}")
        
        # 保存预测结果
        output_test_path = 'house_price_test_predictions.csv'
        result_df.to_csv(output_test_path, index=False, float_format='%.2f')
        print(f"\n预测结果已保存到: {output_test_path}")
        print(f"结果文件包含 {len(result_df)} 条预测记录")
        
        # 显示结果的前几行
        print("\n预测结果前5行:")
        print(result_df.head())
        
        # 显示预测结果的统计信息
        print("\n预测结果统计信息:")
        print(f"预测值 Price 范围: [{result_df['Price'].min():.2f}, {result_df['Price'].max():.2f}]")
        print(f"预测值 Price 均值: {result_df['Price'].mean():.2f}")
        print(f"预测值 Price 标准差: {result_df['Price'].std():.2f}")
        
        return result_df
        
    except Exception as e:
        print(f"预测过程中出现错误: {e}")
        print("尝试诊断问题...")
        print(f"测试集特征维度: {X_test_final_const.shape}")
        print(f"模型参数维度: {len(ols_model.params)}")
        print(f"测试集特征列: {X_test_final_const.columns.tolist()}")
        print(f"模型参数索引: {ols_model.params.index.tolist()}")
        return None

# 执行测试集预测
predict_test_set(model_results)


测试集预测
原始测试集形状: (34017, 62)
训练模型使用的特征数量: 83

执行特征工程...
检查并创建平方项...
已创建平方项: area^2
检查并创建城市哑变量...
已创建城市哑变量: city_0
已创建城市哑变量: city_1
已创建城市哑变量: city_2
已创建城市哑变量: city_3
已创建城市哑变量: city_4
已创建城市哑变量: city_5
已创建城市哑变量: city_6
已创建城市哑变量: city_7
已创建城市哑变量: city_8
已创建城市哑变量: city_9
已创建城市哑变量: city_10
已创建城市哑变量: city_11
检查并创建城市距离变量...
已创建城市距离变量: city_dist_0
已创建城市距离变量: city_dist_1
已创建城市距离变量: city_dist_2
已创建城市距离变量: city_dist_3
已创建城市距离变量: city_dist_4
已创建城市距离变量: city_dist_5
已创建城市距离变量: city_dist_6
已创建城市距离变量: city_dist_7
已创建城市距离变量: city_dist_8
已创建城市距离变量: city_dist_9
已创建城市距离变量: city_dist_10
已创建城市距离变量: city_dist_11
检查并创建交互项...
已创建交互项: city_area_interaction_city_0
已创建交互项: city_area_interaction_city_1
已创建交互项: city_area_interaction_city_2
已创建交互项: city_area_interaction_city_3
已创建交互项: city_area_interaction_city_4
已创建交互项: city_area_interaction_city_5
已创建交互项: city_area_interaction_city_6
已创建交互项: city_area_interaction_city_7
已创建交互项: city_area_interaction_city_8
已创建交互项: city_area_interaction_city_9
已创建交互项: city_area_inter

,ID,Price
0,1000000,1.063529e+07
1,1000001,4.539134e+06
2,1000002,7.491529e+06
3,1000003,2.939031e+06
4,1000004,6.140795e+06
...,...,...
34012,1034012,1.276619e+06
34013,1034013,4.647216e+05
34014,1034014,7.501869e+05
34015,1034015,7.489526e+05


In [8]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Lasso, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# 2.3 LASSO回归建模
print("=" * 60)
print("LASSO回归模型")
print("=" * 60)

# 读取数据
df = pd.read_csv('house_price_train_dataset.csv')
print(f"训练集形状: {df.shape}")

# 创建lnPrice（如果需要）
if 'lnPrice' not in df.columns and 'Price' in df.columns:
    df['lnPrice'] = np.log(df['Price'])

# 设置变量
y_var = 'lnPrice'
X_vars = [col for col in df.columns if col not in ['Price', 'lnPrice', '城市', '区县', '板块']]

print(f"因变量: {y_var}")
print(f"自变量数量: {len(X_vars)}")

# 准备数据
X = df[X_vars]
y = df[y_var]

# 处理缺失值
print("\n处理缺失值...")
X_filled = X.copy()
y_filled = y.copy()

for col in X_filled.columns:
    if X_filled[col].isnull().sum() > 0:
        if X_filled[col].dtype in ['float64', 'int64']:
            X_filled[col] = X_filled[col].fillna(X_filled[col].median())
        else:
            mode_val = X_filled[col].mode()[0] if not X_filled[col].mode().empty else 0
            X_filled[col] = X_filled[col].fillna(mode_val)

if y_filled.isnull().sum() > 0:
    y_filled = y_filled.fillna(y_filled.median())

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_filled, y_filled, test_size=0.2, random_state=42
)
print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")

# 标准化特征
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 使用交叉验证选择最佳alpha值
print("\n寻找最佳alpha值...")
alphas = np.logspace(-4, 2, 50)
lasso_cv = LassoCV(alphas=alphas, cv=5, random_state=42, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)
print(f"最佳alpha: {lasso_cv.alpha_:.6f}")

# 训练最终LASSO模型
lasso_model = Lasso(alpha=lasso_cv.alpha_, random_state=42, max_iter=10000)
lasso_model.fit(X_train_scaled, y_train)

# 预测和评估函数
def predict_and_evaluate(model, X, y, scaler=None, is_scaled=False):
    if not is_scaled and scaler is not None:
        X = scaler.transform(X)
    
    y_pred_ln = model.predict(X)
    y_pred = np.exp(y_pred_ln)
    y_true = np.exp(y)
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    return mae, rmse, r2

# 样本内性能
train_mae, train_rmse, train_r2 = predict_and_evaluate(
    lasso_model, X_train_scaled, y_train, is_scaled=True
)

# 样本外性能
test_mae, test_rmse, test_r2 = predict_and_evaluate(
    lasso_model, X_test, y_test, scaler=scaler
)

# 6折交叉验证
print("\n6折交叉验证...")
kf = KFold(n_splits=6, shuffle=True, random_state=42)
cv_scores_mae, cv_scores_rmse, cv_scores_r2 = [], [], []

for train_idx, val_idx in kf.split(X_filled):
    X_cv_train, X_cv_val = X_filled.iloc[train_idx], X_filled.iloc[val_idx]
    y_cv_train, y_cv_val = y_filled.iloc[train_idx], y_filled.iloc[val_idx]
    
    scaler_cv = StandardScaler()
    X_cv_train_scaled = scaler_cv.fit_transform(X_cv_train)
    X_cv_val_scaled = scaler_cv.transform(X_cv_val)
    
    cv_model = Lasso(alpha=lasso_cv.alpha_, random_state=42, max_iter=10000)
    cv_model.fit(X_cv_train_scaled, y_cv_train)
    
    cv_mae, cv_rmse, cv_r2 = predict_and_evaluate(
        cv_model, X_cv_val_scaled, y_cv_val, is_scaled=True
    )
    
    cv_scores_mae.append(cv_mae)
    cv_scores_rmse.append(cv_rmse)
    cv_scores_r2.append(cv_r2)

cv_mae_mean = np.mean(cv_scores_mae)
cv_rmse_mean = np.mean(cv_scores_rmse)
cv_r2_mean = np.mean(cv_scores_r2)

# 输出结果
print("\n" + "=" * 60)
print("LASSO模型性能")
print("=" * 60)

print(f"{'Metrics':<15} {'In sample':<12} {'Out of sample':<14} {'CV':<12}")
print("-" * 55)
print(f"{'MAE':<15} {train_mae:.2f}{'':<8} {test_mae:.2f}{'':<10} {cv_mae_mean:.2f}")
print(f"{'RMSE':<15} {train_rmse:.2f}{'':<8} {test_rmse:.2f}{'':<10} {cv_rmse_mean:.2f}")
print(f"{'R²':<15} {train_r2:.4f}{'':<8} {test_r2:.4f}{'':<10} {cv_r2_mean:.4f}")
print("-" * 55)

# 系数信息
non_zero_coef = np.sum(lasso_model.coef_ != 0)
print(f"\n非零系数: {non_zero_coef}/{len(lasso_model.coef_)}")

# 保存模型结果
model_results = {
    'LASSO': {
        'model': lasso_model,
        'scaler': scaler,
        'feature_names': X_vars,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'cv_mae': cv_mae_mean,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'cv_rmse': cv_rmse_mean,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'cv_r2': cv_r2_mean,
        'alpha': lasso_cv.alpha_
    }
}

print("\nLASSO模型训练完成")

# 测试集预测函数 - 修复版本
def predict_test_set_lasso(model_results, test_data_path='house_price_test_dataset.csv'):
    print("\n" + "=" * 60)
    print("测试集预测")
    print("=" * 60)
    
    # 读取测试集
    test_df = pd.read_csv(test_data_path)
    print(f"测试集形状: {test_df.shape}")
    
    # 获取模型组件
    lasso_model = model_results['LASSO']['model']
    scaler = model_results['LASSO']['scaler']
    feature_names = model_results['LASSO']['feature_names']
    
    # 特征工程：创建训练集中存在但测试集中缺失的变量
    print("\n执行特征工程...")
    
    # 1. 创建平方项
    if 'area' in test_df.columns and 'area^2' not in test_df.columns:
        test_df['area^2'] = test_df['area'] ** 2
        print("已创建 area^2")
    
    # 2. 创建城市哑变量
    if '城市' in test_df.columns:
        for city_num in range(12):
            city_col = f'city_{city_num}'
            if city_col not in test_df.columns:
                test_df[city_col] = (test_df['城市'] == city_num).astype(int)
                print(f"已创建 {city_col}")
    
    # 3. 创建城市距离变量
    if '城市' in test_df.columns and '距市中心距离_km' in test_df.columns:
        for city_num in range(12):
            city_dist_col = f'city_dist_{city_num}'
            if city_dist_col not in test_df.columns:
                test_df[city_dist_col] = 0
                test_df.loc[test_df['城市'] == city_num, city_dist_col] = test_df.loc[test_df['城市'] == city_num, '距市中心距离_km']
                print(f"已创建 {city_dist_col}")
    
    # 准备测试集特征
    # 检查哪些特征在测试集中存在
    available_features = [col for col in feature_names if col in test_df.columns]
    missing_features = set(feature_names) - set(available_features)
    
    if missing_features:
        print(f"警告: 测试集中缺少 {len(missing_features)} 个特征，将用0填充")
        for feature in missing_features:
            test_df[feature] = 0
    
    X_test = test_df[feature_names].copy()
    
    # 处理缺失值
    print("处理缺失值...")
    for col in X_test.columns:
        if X_test[col].isnull().sum() > 0:
            null_count = X_test[col].isnull().sum()
            if X_test[col].dtype in ['float64', 'int64']:
                X_test[col] = X_test[col].fillna(X_test[col].median())
            else:
                mode_val = X_test[col].mode()[0] if not X_test[col].mode().empty else 0
                X_test[col] = X_test[col].fillna(mode_val)
            print(f"  - {col}: {null_count} 个缺失值已填充")
    
    # 标准化并预测
    X_test_scaled = scaler.transform(X_test)
    y_pred_ln = lasso_model.predict(X_test_scaled)
    y_pred = np.exp(y_pred_ln)
    
    # 创建结果
    if 'ID' in test_df.columns:
        result_df = pd.DataFrame({'ID': test_df['ID'], 'Price': y_pred})
    else:
        result_df = pd.DataFrame({'ID': test_df.index, 'Price': y_pred})
    
    # 保存结果
    output_path = 'house_price_test_predictions_lasso.csv'
    result_df.to_csv(output_path, index=False, float_format='%.2f')
    print(f"预测结果保存至: {output_path}")
    print(f"预测样本数: {len(result_df)}")
    
    # 显示统计信息
    print(f"价格范围: [{result_df['Price'].min():.2f}, {result_df['Price'].max():.2f}]")
    print(f"平均价格: {result_df['Price'].mean():.2f}")
    
    return result_df

# 重新执行测试集预测
predict_test_set_lasso(model_results)

LASSO回归模型
训练集形状: (103871, 88)
因变量: lnPrice
自变量数量: 83

处理缺失值...
训练集: (83096, 83), 测试集: (20775, 83)

寻找最佳alpha值...
最佳alpha: 0.000100

6折交叉验证...

LASSO模型性能
Metrics         In sample    Out of sample  CV          
-------------------------------------------------------
MAE             663907.29         674927.12           666438.33
RMSE            1369229.80         1386343.41           1373679.22
R²              0.7067         0.7049           0.7055
-------------------------------------------------------

非零系数: 78/83

LASSO模型训练完成

测试集预测
测试集形状: (34017, 62)

执行特征工程...
已创建 area^2
已创建 city_0
已创建 city_1
已创建 city_2
已创建 city_3
已创建 city_4
已创建 city_5
已创建 city_6
已创建 city_7
已创建 city_8
已创建 city_9
已创建 city_10
已创建 city_11
已创建 city_dist_0
已创建 city_dist_1
已创建 city_dist_2
已创建 city_dist_3
已创建 city_dist_4
已创建 city_dist_5
已创建 city_dist_6
已创建 city_dist_7
已创建 city_dist_8
已创建 city_dist_9
已创建 city_dist_10
已创建 city_dist_11
处理缺失值...
  - decoration_精装: 14 个缺失值已填充
  - decoration_简装: 14 个缺失值已填充
  - decoration_毛坯: 14

,ID,Price
0,1000000,1.068033e+07
1,1000001,4.518785e+06
2,1000002,7.517455e+06
3,1000003,2.938776e+06
4,1000004,6.150030e+06
...,...,...
34012,1034012,1.235437e+06
34013,1034013,4.682530e+05
34014,1034014,7.418469e+05
34015,1034015,7.406167e+05


In [11]:
import pandas as pd
import numpy as np
from sklearn.linear_model import Ridge, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# 2.4 岭回归建模
print("=" * 60)
print("Ridge回归模型")
print("=" * 60)

# 读取数据
df = pd.read_csv('house_price_train_dataset.csv')
print(f"训练集形状: {df.shape}")

# 创建lnPrice（如果需要）
if 'lnPrice' not in df.columns and 'Price' in df.columns:
    df['lnPrice'] = np.log(df['Price'])

# 设置变量
y_var = 'lnPrice'
X_vars = [col for col in df.columns if col not in ['Price', 'lnPrice', '城市', '区县', '板块']]

print(f"因变量: {y_var}")
print(f"自变量数量: {len(X_vars)}")

# 准备数据
X = df[X_vars]
y = df[y_var]

# 处理缺失值
print("\n处理缺失值...")
X_filled = X.copy()
y_filled = y.copy()

for col in X_filled.columns:
    if X_filled[col].isnull().sum() > 0:
        if X_filled[col].dtype in ['float64', 'int64']:
            X_filled[col] = X_filled[col].fillna(X_filled[col].median())
        else:
            mode_val = X_filled[col].mode()[0] if not X_filled[col].mode().empty else 0
            X_filled[col] = X_filled[col].fillna(mode_val)

if y_filled.isnull().sum() > 0:
    y_filled = y_filled.fillna(y_filled.median())

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_filled, y_filled, test_size=0.2, random_state=42
)
print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")

# 标准化特征
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 使用交叉验证选择最佳alpha值
print("\n寻找最佳alpha值...")
alphas = np.logspace(-2, 4, 50)
ridge_cv = RidgeCV(alphas=alphas, cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train_scaled, y_train)
print(f"最佳alpha: {ridge_cv.alpha_:.6f}")

# 训练最终Ridge模型
ridge_model = Ridge(alpha=ridge_cv.alpha_, random_state=42)
ridge_model.fit(X_train_scaled, y_train)

# 预测和评估函数
def predict_and_evaluate(model, X, y, scaler=None, is_scaled=False):
    if not is_scaled and scaler is not None:
        X = scaler.transform(X)
    
    y_pred_ln = model.predict(X)
    y_pred = np.exp(y_pred_ln)
    y_true = np.exp(y)
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    return mae, rmse, r2

# 样本内性能
train_mae, train_rmse, train_r2 = predict_and_evaluate(
    ridge_model, X_train_scaled, y_train, is_scaled=True
)

# 样本外性能
test_mae, test_rmse, test_r2 = predict_and_evaluate(
    ridge_model, X_test, y_test, scaler=scaler
)

# 6折交叉验证
print("\n6折交叉验证...")
kf = KFold(n_splits=6, shuffle=True, random_state=42)
cv_scores_mae, cv_scores_rmse, cv_scores_r2 = [], [], []

for train_idx, val_idx in kf.split(X_filled):
    X_cv_train, X_cv_val = X_filled.iloc[train_idx], X_filled.iloc[val_idx]
    y_cv_train, y_cv_val = y_filled.iloc[train_idx], y_filled.iloc[val_idx]
    
    scaler_cv = StandardScaler()
    X_cv_train_scaled = scaler_cv.fit_transform(X_cv_train)
    X_cv_val_scaled = scaler_cv.transform(X_cv_val)
    
    cv_model = Ridge(alpha=ridge_cv.alpha_, random_state=42)
    cv_model.fit(X_cv_train_scaled, y_cv_train)
    
    cv_mae, cv_rmse, cv_r2 = predict_and_evaluate(
        cv_model, X_cv_val_scaled, y_cv_val, is_scaled=True
    )
    
    cv_scores_mae.append(cv_mae)
    cv_scores_rmse.append(cv_rmse)
    cv_scores_r2.append(cv_r2)

cv_mae_mean = np.mean(cv_scores_mae)
cv_rmse_mean = np.mean(cv_scores_rmse)
cv_r2_mean = np.mean(cv_scores_r2)

# 输出结果
print("\n" + "=" * 60)
print("Ridge模型性能")
print("=" * 60)

print(f"{'Metrics':<15} {'In sample':<12} {'Out of sample':<14} {'CV':<12}")
print("-" * 55)
print(f"{'MAE':<15} {train_mae:.2f}{'':<8} {test_mae:.2f}{'':<10} {cv_mae_mean:.2f}")
print(f"{'RMSE':<15} {train_rmse:.2f}{'':<8} {test_rmse:.2f}{'':<10} {cv_rmse_mean:.2f}")
print(f"{'R²':<15} {train_r2:.4f}{'':<8} {test_r2:.4f}{'':<10} {cv_r2_mean:.4f}")
print("-" * 55)

# 保存模型结果
model_results = {
    'Ridge': {
        'model': ridge_model,
        'scaler': scaler,
        'feature_names': X_vars,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'cv_mae': cv_mae_mean,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'cv_rmse': cv_rmse_mean,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'cv_r2': cv_r2_mean,
        'alpha': ridge_cv.alpha_
    }
}

print("\nRidge模型训练完成")

# 测试集预测函数 - Ridge模型修复版本
def predict_test_set_ridge(model_results, test_data_path='house_price_test_dataset.csv'):
    print("\n" + "=" * 60)
    print("测试集预测 - Ridge模型")
    print("=" * 60)
    
    # 读取测试集
    test_df = pd.read_csv(test_data_path)
    print(f"测试集形状: {test_df.shape}")
    
    # 获取模型组件
    ridge_model = model_results['Ridge']['model']
    scaler = model_results['Ridge']['scaler']
    feature_names = model_results['Ridge']['feature_names']
    
    # 特征工程：创建训练集中存在但测试集中缺失的变量
    print("\n执行特征工程...")
    
    # 1. 创建平方项
    if 'area' in test_df.columns and 'area^2' not in test_df.columns:
        test_df['area^2'] = test_df['area'] ** 2
        print("已创建 area^2")
    
    # 2. 创建城市哑变量
    if '城市' in test_df.columns:
        for city_num in range(12):
            city_col = f'city_{city_num}'
            if city_col not in test_df.columns:
                test_df[city_col] = (test_df['城市'] == city_num).astype(int)
                print(f"已创建 {city_col}")
    
    # 3. 创建城市距离变量
    if '城市' in test_df.columns and '距市中心距离_km' in test_df.columns:
        for city_num in range(12):
            city_dist_col = f'city_dist_{city_num}'
            if city_dist_col not in test_df.columns:
                test_df[city_dist_col] = 0
                test_df.loc[test_df['城市'] == city_num, city_dist_col] = test_df.loc[test_df['城市'] == city_num, '距市中心距离_km']
                print(f"已创建 {city_dist_col}")
    
    # 准备测试集特征
    # 检查哪些特征在测试集中存在
    available_features = [col for col in feature_names if col in test_df.columns]
    missing_features = set(feature_names) - set(available_features)
    
    if missing_features:
        print(f"警告: 测试集中缺少 {len(missing_features)} 个特征，将用0填充")
        for feature in missing_features:
            test_df[feature] = 0
    
    X_test = test_df[feature_names].copy()
    
    # 处理缺失值
    print("处理缺失值...")
    for col in X_test.columns:
        if X_test[col].isnull().sum() > 0:
            null_count = X_test[col].isnull().sum()
            if X_test[col].dtype in ['float64', 'int64']:
                X_test[col] = X_test[col].fillna(X_test[col].median())
            else:
                mode_val = X_test[col].mode()[0] if not X_test[col].mode().empty else 0
                X_test[col] = X_test[col].fillna(mode_val)
            print(f"  - {col}: {null_count} 个缺失值已填充")
    
    # 标准化并预测
    X_test_scaled = scaler.transform(X_test)
    y_pred_ln = ridge_model.predict(X_test_scaled)
    y_pred = np.exp(y_pred_ln)
    
    # 创建结果
    if 'ID' in test_df.columns:
        result_df = pd.DataFrame({'ID': test_df['ID'], 'Price': y_pred})
    else:
        result_df = pd.DataFrame({'ID': test_df.index, 'Price': y_pred})
    
    # 保存结果
    output_path = 'house_price_test_predictions_ridge.csv'
    result_df.to_csv(output_path, index=False, float_format='%.2f')
    print(f"预测结果保存至: {output_path}")
    print(f"预测样本数: {len(result_df)}")
    
    # 显示统计信息
    print(f"价格范围: [{result_df['Price'].min():.2f}, {result_df['Price'].max():.2f}]")
    print(f"平均价格: {result_df['Price'].mean():.2f}")
    
    return result_df

# 重新执行测试集预测
predict_test_set_ridge(model_results)

Ridge回归模型
训练集形状: (103871, 88)
因变量: lnPrice
自变量数量: 83

处理缺失值...
训练集: (83096, 83), 测试集: (20775, 83)

寻找最佳alpha值...
最佳alpha: 0.910298

6折交叉验证...

Ridge模型性能
Metrics         In sample    Out of sample  CV          
-------------------------------------------------------
MAE             663474.86         674468.12           665964.52
RMSE            1368852.87         1385918.31           1373316.74
R²              0.7068         0.7050           0.7056
-------------------------------------------------------

Ridge模型训练完成

测试集预测 - Ridge模型
测试集形状: (34017, 62)

执行特征工程...
已创建 area^2
已创建 city_0
已创建 city_1
已创建 city_2
已创建 city_3
已创建 city_4
已创建 city_5
已创建 city_6
已创建 city_7
已创建 city_8
已创建 city_9
已创建 city_10
已创建 city_11
已创建 city_dist_0
已创建 city_dist_1
已创建 city_dist_2
已创建 city_dist_3
已创建 city_dist_4
已创建 city_dist_5
已创建 city_dist_6
已创建 city_dist_7
已创建 city_dist_8
已创建 city_dist_9
已创建 city_dist_10
已创建 city_dist_11
处理缺失值...
  - decoration_精装: 14 个缺失值已填充
  - decoration_简装: 14 个缺失值已填充
  - decoration_毛坯: 14 个缺

,ID,Price
0,1000000,1.063343e+07
1,1000001,4.539050e+06
2,1000002,7.491205e+06
3,1000003,2.938812e+06
4,1000004,6.141821e+06
...,...,...
34012,1034012,1.232151e+06
34013,1034013,4.677929e+05
34014,1034014,7.404233e+05
34015,1034015,7.392056e+05


In [13]:
import pandas as pd
import numpy as np
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Elastic Net回归建模
print("=" * 60)
print("Elastic Net回归模型")
print("=" * 60)

# 读取数据
df = pd.read_csv('house_price_train_dataset.csv')
print(f"训练集形状: {df.shape}")

# 创建lnPrice（如果需要）
if 'lnPrice' not in df.columns and 'Price' in df.columns:
    df['lnPrice'] = np.log(df['Price'])

# 设置变量
y_var = 'lnPrice'
X_vars = [col for col in df.columns if col not in ['Price', 'lnPrice', '城市', '区县', '板块']]

print(f"因变量: {y_var}")
print(f"自变量数量: {len(X_vars)}")

# 准备数据
X = df[X_vars]
y = df[y_var]

# 处理缺失值
print("\n处理缺失值...")
X_filled = X.copy()
y_filled = y.copy()

for col in X_filled.columns:
    if X_filled[col].isnull().sum() > 0:
        if X_filled[col].dtype in ['float64', 'int64']:
            X_filled[col] = X_filled[col].fillna(X_filled[col].median())
        else:
            mode_val = X_filled[col].mode()[0] if not X_filled[col].mode().empty else 0
            X_filled[col] = X_filled[col].fillna(mode_val)

if y_filled.isnull().sum() > 0:
    y_filled = y_filled.fillna(y_filled.median())

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X_filled, y_filled, test_size=0.2, random_state=42
)
print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")

# 标准化特征
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 使用交叉验证选择最佳参数
print("\n寻找最佳参数...")
alphas = np.logspace(-3, 1, 20)
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99, 1]

elastic_net_cv = ElasticNetCV(
    alphas=alphas,
    l1_ratio=l1_ratios,
    cv=5,
    random_state=42,
    max_iter=5000,
    n_jobs=-1
)

elastic_net_cv.fit(X_train_scaled, y_train)
print(f"最佳alpha: {elastic_net_cv.alpha_:.6f}")
print(f"最佳l1_ratio: {elastic_net_cv.l1_ratio_:.4f}")

# 训练最终Elastic Net模型
elastic_net_model = ElasticNet(
    alpha=elastic_net_cv.alpha_,
    l1_ratio=elastic_net_cv.l1_ratio_,
    random_state=42,
    max_iter=5000
)
elastic_net_model.fit(X_train_scaled, y_train)

# 预测和评估函数
def predict_and_evaluate(model, X, y, scaler=None, is_scaled=False):
    if not is_scaled and scaler is not None:
        X = scaler.transform(X)
    
    y_pred_ln = model.predict(X)
    y_pred = np.exp(y_pred_ln)
    y_true = np.exp(y)
    
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    return mae, rmse, r2

# 样本内性能
train_mae, train_rmse, train_r2 = predict_and_evaluate(
    elastic_net_model, X_train_scaled, y_train, is_scaled=True
)

# 样本外性能
test_mae, test_rmse, test_r2 = predict_and_evaluate(
    elastic_net_model, X_test, y_test, scaler=scaler
)

# 6折交叉验证
print("\n6折交叉验证...")
kf = KFold(n_splits=6, shuffle=True, random_state=42)
cv_scores_mae, cv_scores_rmse, cv_scores_r2 = [], [], []

for train_idx, val_idx in kf.split(X_filled):
    X_cv_train, X_cv_val = X_filled.iloc[train_idx], X_filled.iloc[val_idx]
    y_cv_train, y_cv_val = y_filled.iloc[train_idx], y_filled.iloc[val_idx]
    
    scaler_cv = StandardScaler()
    X_cv_train_scaled = scaler_cv.fit_transform(X_cv_train)
    X_cv_val_scaled = scaler_cv.transform(X_cv_val)
    
    cv_model = ElasticNet(
        alpha=elastic_net_cv.alpha_,
        l1_ratio=elastic_net_cv.l1_ratio_,
        random_state=42,
        max_iter=5000
    )
    cv_model.fit(X_cv_train_scaled, y_cv_train)
    
    cv_mae, cv_rmse, cv_r2 = predict_and_evaluate(
        cv_model, X_cv_val_scaled, y_cv_val, is_scaled=True
    )
    
    cv_scores_mae.append(cv_mae)
    cv_scores_rmse.append(cv_rmse)
    cv_scores_r2.append(cv_r2)

cv_mae_mean = np.mean(cv_scores_mae)
cv_rmse_mean = np.mean(cv_scores_rmse)
cv_r2_mean = np.mean(cv_scores_r2)

# 输出结果
print("\n" + "=" * 60)
print("Elastic Net模型性能")
print("=" * 60)

print(f"{'Metrics':<15} {'In sample':<12} {'Out of sample':<14} {'CV':<12}")
print("-" * 55)
print(f"{'MAE':<15} {train_mae:.2f}{'':<8} {test_mae:.2f}{'':<10} {cv_mae_mean:.2f}")
print(f"{'RMSE':<15} {train_rmse:.2f}{'':<8} {test_rmse:.2f}{'':<10} {cv_rmse_mean:.2f}")
print(f"{'R²':<15} {train_r2:.4f}{'':<8} {test_r2:.4f}{'':<10} {cv_r2_mean:.4f}")
print("-" * 55)

# 系数信息
non_zero_coef = np.sum(elastic_net_model.coef_ != 0)
print(f"\n非零系数: {non_zero_coef}/{len(elastic_net_model.coef_)}")

# 保存模型结果
model_results = {
    'ElasticNet': {
        'model': elastic_net_model,
        'scaler': scaler,
        'feature_names': X_vars,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'cv_mae': cv_mae_mean,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'cv_rmse': cv_rmse_mean,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'cv_r2': cv_r2_mean,
        'alpha': elastic_net_cv.alpha_,
        'l1_ratio': elastic_net_cv.l1_ratio_
    }
}

print("\nElastic Net模型训练完成")

# 测试集预测函数 - Elastic Net模型修复版本
def predict_test_set_elastic_net(model_results, test_data_path='house_price_test_dataset.csv'):
    print("\n" + "=" * 60)
    print("测试集预测 - Elastic Net模型")
    print("=" * 60)
    
    # 读取测试集
    test_df = pd.read_csv(test_data_path)
    print(f"测试集形状: {test_df.shape}")
    
    # 获取模型组件
    elastic_net_model = model_results['ElasticNet']['model']
    scaler = model_results['ElasticNet']['scaler']
    feature_names = model_results['ElasticNet']['feature_names']
    
    # 特征工程：创建训练集中存在但测试集中缺失的变量
    print("\n执行特征工程...")
    
    # 1. 创建平方项
    if 'area' in test_df.columns and 'area^2' not in test_df.columns:
        test_df['area^2'] = test_df['area'] ** 2
        print("已创建 area^2")
    
    # 2. 创建城市哑变量
    if '城市' in test_df.columns:
        for city_num in range(12):
            city_col = f'city_{city_num}'
            if city_col not in test_df.columns:
                test_df[city_col] = (test_df['城市'] == city_num).astype(int)
                print(f"已创建 {city_col}")
    
    # 3. 创建城市距离变量
    if '城市' in test_df.columns and '距市中心距离_km' in test_df.columns:
        for city_num in range(12):
            city_dist_col = f'city_dist_{city_num}'
            if city_dist_col not in test_df.columns:
                test_df[city_dist_col] = 0
                test_df.loc[test_df['城市'] == city_num, city_dist_col] = test_df.loc[test_df['城市'] == city_num, '距市中心距离_km']
                print(f"已创建 {city_dist_col}")
    
    # 准备测试集特征
    # 检查哪些特征在测试集中存在
    available_features = [col for col in feature_names if col in test_df.columns]
    missing_features = set(feature_names) - set(available_features)
    
    if missing_features:
        print(f"警告: 测试集中缺少 {len(missing_features)} 个特征，将用0填充")
        for feature in missing_features:
            test_df[feature] = 0
    
    X_test = test_df[feature_names].copy()
    
    # 处理缺失值
    print("处理缺失值...")
    for col in X_test.columns:
        if X_test[col].isnull().sum() > 0:
            null_count = X_test[col].isnull().sum()
            if X_test[col].dtype in ['float64', 'int64']:
                X_test[col] = X_test[col].fillna(X_test[col].median())
            else:
                mode_val = X_test[col].mode()[0] if not X_test[col].mode().empty else 0
                X_test[col] = X_test[col].fillna(mode_val)
            print(f"  - {col}: {null_count} 个缺失值已填充")
    
    # 标准化并预测
    X_test_scaled = scaler.transform(X_test)
    y_pred_ln = elastic_net_model.predict(X_test_scaled)
    y_pred = np.exp(y_pred_ln)
    
    # 创建结果
    if 'ID' in test_df.columns:
        result_df = pd.DataFrame({'ID': test_df['ID'], 'Price': y_pred})
    else:
        result_df = pd.DataFrame({'ID': test_df.index, 'Price': y_pred})
    
    # 保存结果
    output_path = 'house_price_test_predictions_elastic_net.csv'
    result_df.to_csv(output_path, index=False, float_format='%.2f')
    print(f"预测结果保存至: {output_path}")
    print(f"预测样本数: {len(result_df)}")
    
    # 显示统计信息
    print(f"价格范围: [{result_df['Price'].min():.2f}, {result_df['Price'].max():.2f}]")
    print(f"平均价格: {result_df['Price'].mean():.2f}")
    
    return result_df

# 重新执行测试集预测
predict_test_set_elastic_net(model_results)

Elastic Net回归模型
训练集形状: (103871, 88)
因变量: lnPrice
自变量数量: 83

处理缺失值...
训练集: (83096, 83), 测试集: (20775, 83)

寻找最佳参数...
最佳alpha: 0.001000
最佳l1_ratio: 0.1000

6折交叉验证...

Elastic Net模型性能
Metrics         In sample    Out of sample  CV          
-------------------------------------------------------
MAE             665083.49         676317.34           667415.83
RMSE            1368952.30         1385739.54           1372885.33
R²              0.7068         0.7051           0.7058
-------------------------------------------------------

非零系数: 79/83

Elastic Net模型训练完成

测试集预测 - Elastic Net模型
测试集形状: (34017, 62)

执行特征工程...
已创建 area^2
已创建 city_0
已创建 city_1
已创建 city_2
已创建 city_3
已创建 city_4
已创建 city_5
已创建 city_6
已创建 city_7
已创建 city_8
已创建 city_9
已创建 city_10
已创建 city_11
已创建 city_dist_0
已创建 city_dist_1
已创建 city_dist_2
已创建 city_dist_3
已创建 city_dist_4
已创建 city_dist_5
已创建 city_dist_6
已创建 city_dist_7
已创建 city_dist_8
已创建 city_dist_9
已创建 city_dist_10
已创建 city_dist_11
处理缺失值...
  - decoration_精装: 14 个缺失值已填充
  

,ID,Price
0,1000000,1.066711e+07
1,1000001,4.521006e+06
2,1000002,7.480983e+06
3,1000003,2.918826e+06
4,1000004,6.210847e+06
...,...,...
34012,1034012,1.275126e+06
34013,1034013,4.642979e+05
34014,1034014,7.680525e+05
34015,1034015,7.668122e+05
